# From Static Prompts to LangChain Chains with Groq

**Bano Qabil — Beginner Classroom Notebook**

This lesson starts with a prompt written directly in Python and gradually converts it into a reusable LangChain chain. The examples use the **Groq API** through `ChatGroq`.

### Learning goals

By the end, you will be able to:

1. Explain the difference between a system message and a user message.
2. Call a Groq-hosted chat model through LangChain.
3. Replace fixed text with variables in `ChatPromptTemplate`.
4. Build a chain with the LCEL pipe operator `|`.
5. Reuse one chain with different inputs.
6. Connect two small chains to make a simple workflow.

## Step 0 — VS Code / local Jupyter setup

In VS Code, select a Python kernel from the top-right corner, then run cells in order. Keep `.env` private and do not upload it to GitHub.

In [ ]:
%pip install -qU langchain-core langchain-groq python-dotenv ipykernel


In [ ]:
# load_dotenv() reads private values from a local .env file.
# Create .env beside this notebook and add: GROQ_API_KEY=your_key_here
import os
from getpass import getpass
from dotenv import load_dotenv

load_dotenv()
groq_key = os.getenv("GROQ_API_KEY")

# Friendly fallback for a classroom computer without a .env file.
if not groq_key:
    groq_key = getpass("Paste your Groq API key: ")
    os.environ["GROQ_API_KEY"] = groq_key

if not groq_key:
    raise ValueError("GROQ_API_KEY was not provided.")

print("Groq API key is ready.")


## Step 1 — Import the building blocks

Think of our application as a small pipeline:

**Prompt → Groq model → plain text output**

`ChatPromptTemplate` prepares messages, `ChatGroq` talks to the model, and `StrOutputParser` gives us a normal Python string.

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda

# One model setting is reused throughout the notebook.
# This model ID is in Groq's catalog. ChatGroq + GROQ_API_KEY mean
# the request is sent through Groq even though the ID contains a vendor name.
# temperature=0 keeps classroom demonstrations more consistent.
MODEL_NAME = "openai/gpt-oss-20b"
llm = ChatGroq(model=MODEL_NAME, temperature=0)
parser = StrOutputParser()

print("Model connected through Groq:", MODEL_NAME)


## Step 2 — First static prompt

A **static prompt** has fixed text. It is useful for learning, but changing the task means editing the code.

- **System message:** sets the role, rules, tone, or boundaries.
- **User message:** contains the question or task.

In [ ]:
# These messages are fixed: there are no variables yet.
static_messages = [
    ("system", "You are a friendly Generative AI teacher. Explain in very simple English and add one short Roman Urdu line."),
    ("human", "Explain an API using a restaurant waiter analogy in 5 lines."),
]

response = llm.invoke(static_messages)
print(response.content)


### Observe: what does the model return?

`llm.invoke(...)` returns an AI message object. Its generated text is stored in `.content`. Later, the output parser will remove this extra step for us.

In [ ]:
print("Python type:", type(response))
print("Only the text:", response.content)


## Step 3 — Change the instructions and compare

The topic below is different from the original repository example. Change only the system role and notice how the same user topic can be explained for a different audience.

In [ ]:
comparison_messages = [
    ("system", "You are a patient teacher for a 15-year-old beginner. Use one everyday analogy and no difficult words."),
    ("human", "What is the difference between a prompt and a model?"),
]

comparison_response = llm.invoke(comparison_messages)
print(comparison_response.content)


## Step 4 — Why static prompts become a problem

Suppose 100 students ask about different topics. Copying and editing the whole prompt 100 times is error-prone. We need placeholders such as `{topic}`, `{audience}`, and `{language}`.

In [ ]:
# A template is like a reusable form. Values will be inserted later.
teaching_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a helpful teacher for {audience}. Explain in {language}. "
     "Use simple words, one analogy, and one practical example."),
    ("human", "Teach me this topic: {topic}"),
])

print(teaching_prompt)


## Step 5 — Fill the template without calling the model

This is an important debugging habit: inspect the final messages first. It costs no API call and helps you catch missing or incorrect variables.

In [ ]:
filled_messages = teaching_prompt.invoke({
    "audience": "beginner Python students",
    "language": "simple English with key terms in Roman Urdu",
    "topic": "temperature in an LLM",
})

print(filled_messages.to_string())


## Step 6 — Run the pieces manually

Before making a chain, run each part yourself. This makes the data flow easy to see.

In [ ]:
# 1) The prompt creates messages.
prompt_value = teaching_prompt.invoke({
    "audience": "first-semester university students",
    "language": "simple English",
    "topic": "tokens in a language model",
})

# 2) The model reads those messages and returns an AI message.
ai_message = llm.invoke(prompt_value)

# 3) The parser converts the AI message into a plain string.
plain_text = parser.invoke(ai_message)

print(plain_text)


## Step 7 — Convert the same flow into an LCEL chain

LCEL means **LangChain Expression Language**. The `|` symbol passes the output of one component into the next component. It is similar to a pipe carrying water forward.

In [ ]:
# Read this line from left to right:
# input dictionary -> formatted prompt -> Groq model -> Python string
teaching_chain = teaching_prompt | llm | parser

answer = teaching_chain.invoke({
    "audience": "Bano Qabil beginners",
    "language": "simple English and Roman Urdu",
    "topic": "why LangChain is useful",
})

print(answer)


## Step 8 — Reuse the same chain

The chain is now reusable. We change only the input dictionary; the pipeline stays the same.

In [ ]:
class_topics = [
    "system prompt versus user prompt",
    "few-shot prompting",
    "hallucination in an LLM",
]

for topic in class_topics:
    result = teaching_chain.invoke({
        "audience": "beginner Generative AI students",
        "language": "simple English",
        "topic": topic,
    })
    print(f"\n--- {topic.upper()} ---")
    print(result)


## Step 9 — Add our own Python function as a chain step

A chain is not limited to prompts and models. A normal Python function can clean, label, or save the result. `RunnableLambda` turns our function into a LangChain component.

In [ ]:
def add_classroom_heading(text: str) -> str:
    """Add a consistent heading to every generated lesson."""
    return "BANO QABIL — QUICK STUDY NOTE\n" + "=" * 35 + "\n" + text.strip()

heading_step = RunnableLambda(add_classroom_heading)
formatted_chain = teaching_prompt | llm | parser | heading_step

formatted_answer = formatted_chain.invoke({
    "audience": "students who know basic Python",
    "language": "simple English",
    "topic": "prompt templates",
})

print(formatted_answer)


## Step 10 — Build a simple two-stage chain

Now one chain writes a short lesson. A second prompt receives that lesson and creates a quiz.

**Topic → Lesson chain → Quiz prompt → Groq model → Quiz text**

In [ ]:
lesson_prompt = ChatPromptTemplate.from_messages([
    ("system", "You teach Generative AI to beginners. Use simple English and stay under 120 words."),
    ("human", "Write a short lesson about {topic}. Include one analogy and one example."),
])

quiz_prompt = ChatPromptTemplate.from_messages([
    ("system", "You create easy classroom quizzes. Use only the lesson supplied by the user."),
    ("human",
     "LESSON:\n{lesson}\n\nCreate 3 multiple-choice questions. "
     "Give options A, B, C and write an answer key at the end."),
])

lesson_chain = lesson_prompt | llm | parser

# The dictionary sends lesson_chain's text into the {lesson} placeholder.
lesson_to_quiz_chain = {"lesson": lesson_chain} | quiz_prompt | llm | parser

quiz = lesson_to_quiz_chain.invoke({"topic": "static prompts and dynamic prompts"})
print(quiz)


## Step 11 — Turn the chain into a tiny interactive app

This final cell accepts student input. It shows that the same chain can sit behind a website, chatbot, or mobile app later.

In [ ]:
student_topic = input("Enter a Generative AI topic: " ).strip()
student_language = input("Choose language (English / Urdu / Roman Urdu): " ).strip()

if not student_topic:
    student_topic = "prompt engineering"
if not student_language:
    student_language = "simple English"

student_note = formatted_chain.invoke({
    "audience": "beginner students",
    "language": student_language,
    "topic": student_topic,
})

print("\n" + student_note)


## Practice tasks

1. Change the system message so the model acts as an electronics teacher.
2. Add a `{length}` variable and request either 5 lines or 10 lines.
3. Make the teaching chain explain **embeddings** in Roman Urdu.
4. Change the custom Python function so it adds your own institute name.
5. Extend the two-stage chain so it produces 5 quiz questions instead of 3.

### Key idea

A prompt is one instruction. A template makes the instruction reusable. A chain connects prompting, the model, parsing, and other steps into one repeatable workflow.

### References

- [Original beginner notebook used as the structural starting point](https://github.com/techsimpluslearning/Complete-GenAI-Course-Python-Langchain/blob/main/notebooks/2_prompt_cains.ipynb)
- [Groq supported models](https://console.groq.com/docs/models)
- [LangChain ChatGroq integration](https://docs.langchain.com/oss/python/integrations/chat/groq)